# BanglaBayanno: Colab/Kaggle Runner\n
\n
This notebook is a ready-to-run launcher for this repository on **Google Colab** or **Kaggle**.\n
\n
- Default mode uses Hugging Face local model path (`qwen2-vl` with `--load_in_4bit`).\n
- Optional mode uses Ollama (`ollama-qwen2.5vl`), but it needs much higher memory.\n
\n
Run cells top to bottom.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import json
import time

# ======================\n
# User config\n
# ======================\n
REPO_URL = "https://github.com/your-username/your-repo.git"
PROJECT_DIR = Path("/content/banglaDataset_CVPR") if Path("/content").exists() else Path("/kaggle/working/banglaDataset_CVPR")\n

# Choose: "hf_local" or "ollama"\n
RUN_MODE = "hf_local"

# Evaluation parameters\n
MODEL_NAME = "qwen2-vl" if RUN_MODE == "hf_local" else "ollama-qwen2.5vl"
SAMPLE_SIZE = 20
REQUEST_DELAY = 0.0
MAX_RETRIES = 3
RETRY_BACKOFF = 2.0

# Set True only if data/qa.json and data/images are not already present\n
DOWNLOAD_DATASET = False

print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_MODE:", RUN_MODE)
print("MODEL_NAME:", MODEL_NAME)

In [ ]:
def run(cmd: str, cwd: Path | None = None) -> None:
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Clone repo if missing\n
if not PROJECT_DIR.exists():\n
    if "your-username/your-repo" in REPO_URL:\n
        raise ValueError("Set REPO_URL to your repository URL first.")\n
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(PROJECT_DIR))}")\n
\n
os.chdir(PROJECT_DIR)\n
print("Working directory:", Path.cwd())

In [ ]:
# Install dependencies\n
run("python -m pip install --upgrade pip")\n
\n
if RUN_MODE == "hf_local":\n
    run('python -m pip install -e ".[local]"', cwd=PROJECT_DIR)\n
else:\n
    run("python -m pip install -e .", cwd=PROJECT_DIR)\n
    run("python -m pip install ollama", cwd=PROJECT_DIR)

In [ ]:
# Dataset check/download\n
qa_file = PROJECT_DIR / "data" / "qa.json"\n
images_dir = PROJECT_DIR / "data" / "images"\n
\n
if qa_file.exists() and images_dir.exists():\n
    print("Dataset found:", qa_file, images_dir)\n
elif DOWNLOAD_DATASET:\n
    run("bash scripts/download_dataset.sh", cwd=PROJECT_DIR)\n
else:\n
    raise FileNotFoundError(\n
        "Dataset not found. Place data/qa.json and data/images, or set DOWNLOAD_DATASET=True."\n
    )

In [ ]:
# Optional Ollama setup (only when RUN_MODE == "ollama")\n
if RUN_MODE == "ollama":\n
    run("curl -fsSL https://ollama.com/install.sh | sh")\n
    _ollama_proc = subprocess.Popen("ollama serve", shell=True)\n
    time.sleep(5)\n
    run("ollama pull qwen2.5vl:7b")\n
    run("ollama list")\n
    print("Ollama server pid:", _ollama_proc.pid)\n
else:\n
    print("Skipping Ollama setup.")

In [ ]:
# Run benchmark\n
cmd = [\n
    "python", "evaluate.py",\n
    "--model", MODEL_NAME,\n
    "--sample", str(SAMPLE_SIZE),\n
    "--request_delay", str(REQUEST_DELAY),\n
    "--max_retries", str(MAX_RETRIES),\n
    "--retry_backoff", str(RETRY_BACKOFF),\n
]\n
\n
if RUN_MODE == "hf_local":\n
    cmd.append("--load_in_4bit")\n
\n
run(" \

: 
,
: null,
: {},
: [],
: [
,
results"\n
metrics_files = sorted(results_dir.glob("*_metrics.json"), key=lambda p: p.stat().st_mtime)\n
\n
if not metrics_files:\n
    raise FileNotFoundError("No metrics file found in results/.")\n
\n
latest = metrics_files[-1]\n
print("Latest metrics:", latest.name)\n
print(json.dumps(json.loads(latest.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

In [ ]:
# Optional: generate figures\n
run("python visualize.py --results_dir results --output figures", cwd=PROJECT_DIR)\n
print("Figures generated in figures/")